## NumPy warm-up

In [1]:
#Let's learn some NumPy and BERT!

In [2]:
#01
import numpy as np

In [3]:
#02
l = [1, 2, 4, 10]
a = np.array(l)
a

array([ 1,  2,  4, 10])

In [4]:
#03
a.shape

(4,)

In [5]:
#04
a[0]
a[1]
a[3]

np.int64(10)

In [6]:
#05
a[3] = 4
a

array([1, 2, 4, 4])

In [7]:
#06
a.sum()

np.int64(11)

In [8]:
#07
a.mean()

np.float64(2.75)

In [9]:
#08
lol = [[1,2,3,4],[5,6,7,8],[9,10,11,12]]
m = np.array(lol)
m

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12]])

In [10]:
#09
m.dot(m.T)

array([[ 30,  70, 110],
       [ 70, 174, 278],
       [110, 278, 446]])

In [11]:
#10
m[1]

array([5, 6, 7, 8])

In [12]:
#11
m[:,1]

array([ 2,  6, 10])

## BERT: token-level embeddings

### Installing transformers and loading LLMs

In [13]:
#12 
!pip install transformers
import transformers

In [14]:
#13
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')
model = BertModel.from_pretrained("bert-base-cased")

### Tokenization

In [15]:
#14
text = "We have learned to use numpy."
encoded_input = tokenizer(text, return_tensors='pt')

In [16]:
#15
encoded_input

{'input_ids': tensor([[  101,  1284,  1138,  3560,  1106,  1329,   183, 15629,  1183,   119,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [17]:
#16
encoded_input['input_ids']

tensor([[  101,  1284,  1138,  3560,  1106,  1329,   183, 15629,  1183,   119,
           102]])

In [18]:
#17
encoded_input['input_ids'].numpy()

array([[  101,  1284,  1138,  3560,  1106,  1329,   183, 15629,  1183,
          119,   102]])

In [19]:
#18
encoded_input['input_ids'].numpy()[0]

array([  101,  1284,  1138,  3560,  1106,  1329,   183, 15629,  1183,
         119,   102])

In [20]:
#19
for i in encoded_input['input_ids'].numpy()[0]:
    print(i, tokenizer.decode(i))

101 [CLS]
1284 We
1138 have
3560 learned
1106 to
1329 use
183 n
15629 ##ump
1183 ##y
119 .
102 [SEP]


### Computing contextual vectors

In [21]:
#20
output = model(**encoded_input)

In [22]:
#21
encoded_input['input_ids'].shape, output['last_hidden_state'].shape

(torch.Size([1, 11]), torch.Size([1, 11, 768]))

In [23]:
#22
M = output['last_hidden_state'].detach().numpy()

In [24]:
#23
M[:,10,:][0]

array([ 7.34819412e-01,  3.93299520e-01, -4.21728976e-02,  1.13882351e+00,
        4.22986448e-01, -7.85803676e-01, -5.89289427e-01,  2.92102247e-01,
        4.34446149e-02,  5.81990778e-02,  1.47634372e-02, -2.48073280e-01,
       -8.50721300e-01,  7.60459244e-01, -1.15883994e+00, -6.97432399e-01,
       -5.14148295e-01,  4.10395712e-01,  4.06388998e-01,  6.56115234e-01,
        5.33256650e-01,  1.11080408e-01, -6.93445146e-01, -6.25239015e-01,
        3.83777022e-01, -6.84640408e-01,  4.48468029e-01,  2.00372338e+00,
        3.93532515e-01,  8.93754065e-01,  6.27744913e-01,  4.91565645e-01,
       -4.48721588e-01, -2.75222629e-01, -5.23494065e-01, -7.13070810e-01,
       -5.01207471e-01,  2.99868494e-01, -8.43545496e-01, -3.40774387e-01,
       -1.10865608e-01,  1.32081568e-01,  2.58419931e-01, -1.60163611e-01,
        9.62146044e-01, -1.08769548e+00, -1.16238737e+00,  2.89523751e-01,
       -8.37998748e-01,  5.37335753e-01,  3.86432931e-02, -8.73039544e-01,
       -1.95636362e-01,  

### Retrieving a specific token

Put in np matrix

In [25]:
#24
sentences = ['The dog can bark loudly.', 
             'He scraped the bark off.',
             'The bark of this tree is not healthy.']
bark_vectors = []
for s in sentences:
    encoded_input = tokenizer(s, return_tensors='pt')

    bark_token = next((i for i,encoding in enumerate(encoded_input['input_ids'][0])
                       if tokenizer.decode(encoding).startswith('bark')),None)
    print(bark_token)
    output = model(**encoded_input)
    vectors = output['last_hidden_state'].detach().numpy()
    bark_vector = vectors[:,bark_token,:][0]
    bark_vectors.append(bark_vector)

4
4
2


In [26]:
#25
bark_vectors = np.array(bark_vectors)

### Cosine similarity

Nearest neighbors

In [27]:
#26
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity(bark_vectors)

array([[0.9999999 , 0.7330581 , 0.72528684],
       [0.7330581 , 0.9999999 , 0.85205954],
       [0.72528684, 0.85205954, 1.        ]], dtype=float32)

## Lexical Substitution

### Another model: GPT2

In [28]:
#27
from transformers import GPT2Tokenizer, GPT2Model, GPT2LMHeadModel
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [30]:
#28
import torch

In [31]:
#29
# from: https://github.com/tmalsburg/llm_surprisal/blob/main/llm_generate.py

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch.nn.functional as F

def surprisal(input_text):
  input_tokens = tokenizer.encode(input_text, return_tensors='pt').to(device)
  with torch.no_grad():
    outputs = model(input_tokens, labels=input_tokens)
    logits = outputs.logits

  # Compute log probabilities for all tokens:
  log_probs = F.log_softmax(logits, dim=-1)

  # Shift input_tokens to get surprisal for all tokens, including
  # the first one:
  shifted_tokens = input_tokens[..., 1:]

  # Gather the log probabilities of target tokens:
  target_log_probs = log_probs[:, :-1].gather(2, shifted_tokens.unsqueeze(-1)).squeeze(-1)

  # Compute surprisal (-log probability):
  surprisals = -target_log_probs / torch.log(torch.tensor(2.0))

  # Handle the first token separately:
  first_token = input_tokens[0, 0].item()
  first_token_surprisal = (-log_probs[0, 0, first_token].item() / torch.log(torch.tensor(2.0))).item()

  # Convert token IDs to readable tokens:
  # Note: tolist does not return a list when there's just one token,
  # but a plain int.
  it = input_tokens.squeeze().tolist()
  if type(it) == int:
    it = [it]
  decoded_tokens = [tokenizer.decode([tok]) for tok in it]

  # Include the first token's surprisal:
  surprisals = [first_token_surprisal] + surprisals.squeeze(0).tolist()

  return list(zip(decoded_tokens, surprisals))

In [32]:
#30
surprisal("I read it in an ancient text.")

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


[('I', 8.554031372070312),
 (' read', 11.177523612976074),
 (' it', 4.085540771484375),
 (' in', 3.754472494125366),
 (' an', 5.568809986114502),
 (' ancient', 9.709693908691406),
 (' text', 4.518036365509033),
 ('.', 3.3201258182525635)]

In [33]:
#31
surprisal("I stirred the soup with my text.")

[('I', 8.554031372070312),
 (' stirred', 17.33550453186035),
 (' the', 2.796382427215576),
 (' soup', 6.767242908477783),
 (' with', 3.0745091438293457),
 (' my', 3.1949031352996826),
 (' text', 17.07008934020996),
 ('.', 5.656049728393555)]